# Protocolo Experimental: Determinación de la Constante de Planck

Este documento contiene el flujo de trabajo para caracterizar la curva de encendido de diodos LED y calcular la constante de Planck.

Teoría y Montaje

La energía del fotón emitido por un LED está relacionada con el voltaje de codo por la ecuación h = (e * U_0 * lambda) / c.
Se requiere un Arduino Uno, un potenciómetro de 5k ohms para variar el voltaje gradualmente y una resistencia fija de 330 ohms para medir la corriente indirectamente.
Conecte el potenciómetro como divisor de tensión y use las entradas analógicas para medir el voltaje total y el voltaje en el diodo.

In [ ]:
import serial
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import os

e = 1.602e-19
c = 299792458
h_teorico = 6.62607015e-34
RESISTENCIA = 330

leds = {
    'rojo':     {'color_graf': 'red',    'lambda': 654.5e-9},
    'blanco':   {'color_graf': 'silver', 'lambda': 450.0e-9},
    'verde':    {'color_graf': 'green',  'lambda': 525.0e-9}
}

PUERTO = 'COM3'
BAUD_RATE = 9600
PUNTOS_POR_CURVA = 60

Captura de Datos

El bloque inferior abre el puerto serial y espera a que el potenciómetro se mueva para empezar a registrar los puntos de la curva de voltaje y corriente.

In [ ]:
def recolectar_curva(color, experimento):
    archivo = f'datos_{color}_{experimento}.csv'
    try:
        with serial.Serial(PUERTO, BAUD_RATE, timeout=1) as ser:
            time.sleep(2)
            print(f"Preparando {color} - Intento {experimento}. Gire el potenciómetro a 0V.")
            input("Presione ENTER y gire lentamente hacia 5V...")
            
            datos = []
            voltaje_inicial = None
            
            while voltaje_inicial is None:
                if ser.in_waiting > 0:
                    try:
                        linea = ser.readline().decode('utf-8').strip()
                        v, i = map(float, linea.split(','))
                        voltaje_inicial = v
                    except:
                        pass
            
            print("Esperando movimiento...")
            grabando = False
            
            while len(datos) < PUNTOS_POR_CURVA:
                if ser.in_waiting > 0:
                    try:
                        linea = ser.readline().decode('utf-8').strip()
                        v, i = map(float, linea.split(','))
                        
                        if not grabando and abs(v - voltaje_inicial) >= 0.05:
                            grabando = True
                            print("Grabando...")
                            
                        if grabando:
                            datos.append([len(datos)+1, v, i])
                            time.sleep(0.2)
                    except:
                        pass
                        
            df = pd.DataFrame(datos, columns=['Punto', 'Voltaje (V)', 'Corriente (mA)'])
            df.to_csv(archivo, index=False)
            print(f"Guardado en {archivo}")
    except Exception as e:
        print(f"Error de puerto: {e}")

# Para usarlo, descomente la siguiente linea y cambie los parametros
# recolectar_curva('rojo', 1)

Procesamiento y Ajuste Lineal

Este script lee todos los archivos generados, aplica un filtro para detectar el codo exponencial y traza la extrapolación lineal al eje X para encontrar el voltaje de encendido.

In [ ]:
NUM_EXPERIMENTOS = 10
plt.figure(figsize=(7, 8))
h_globales = []

for color_nombre, config in leds.items():
    valores_v0 = []
    valores_h = []
    V_acumulados = []
    I_acumulados = []
    
    for exp in range(1, NUM_EXPERIMENTOS + 1):
        try:
            df = pd.read_csv(f'datos_{color_nombre}_{exp}.csv')
            V = df['Voltaje (V)'].values
            I = df['Corriente (mA)'].values
            
            V_acumulados.extend(V)
            I_acumulados.extend(I)
            
            plt.scatter(V, I, color=config['color_graf'], alpha=0.15, s=20)
            
            indices = np.where(I > 0.05)[0]
            if len(indices) < 2:
                continue
                
            V_aj = V[indices]
            I_aj = I[indices]
            
            pendiente, interseccion, _, _, _ = linregress(V_aj, I_aj)
            V0 = -interseccion / pendiente 
            valores_v0.append(V0)
            
            h_exp = (e * V0 * config['lambda']) / c
            valores_h.append(h_exp)
            
        except FileNotFoundError:
            pass 

    if len(valores_v0) > 0:
        v0_promedio = np.mean(valores_v0)
        h_promedio = np.mean(valores_h)
        h_globales.extend(valores_h)
        
        print(f"\n--- RESULTADOS LED {color_nombre.upper()} ---")
        print(f"U_0 Promedio: {v0_promedio:.3f} V")
        print(f"h calculada : {h_promedio:.8e} J.s")
        
        V_arr = np.array(V_acumulados)
        I_arr = np.array(I_acumulados)
        idx = np.where(I_arr > 0.05)[0]
        
        if len(idx) > 2:
            p, i, _, _, _ = linregress(V_arr[idx], I_arr[idx])
            V_line = np.linspace(v0_promedio, max(V_arr), 100)
            I_line = p * V_line + i
            
            plt.plot(V_line, I_line, color=config['color_graf'], linewidth=2, label=f'{color_nombre.capitalize()} ($U_0$={v0_promedio:.2f} V)')
            plt.plot([v0_promedio, v0_promedio], [0, p*v0_promedio + i], color=config['color_graf'], linestyle='--')

if len(h_globales) > 0:
    h_final_prom = np.mean(h_globales)
    h_final_std = np.std(h_globales)
    err = abs(h_final_prom - h_teorico) / h_teorico * 100
    
    print("\n=== CÁLCULO FINAL DE LA CONSTANTE DE PLANCK ===")
    print(f"\u0305h experimental: ({h_final_prom:.8e} \u00B1 {h_final_std:.8e}) J.s")
    print(f"Error relativo total: {err:.2f}%")
    
    texto_info = (f"Resistencia: {RESISTENCIA} $\\Omega$\n"
                  f"$\\overline{{h}}$ exp: {h_final_prom:.3e} J$\\cdot$s\n"
                  f"Error rel: {err:.2f}%")
    plt.text(1.52, max(I_acumulados)*0.85, texto_info, fontsize=10, 
             bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

plt.axhline(0, color='black', linewidth=1.2) 
plt.title("Curvas Características I-V de LEDs (Rojo, Blanco, Verde)")
plt.xlabel("Voltaje (V)") 
plt.ylabel("Corriente (mA)")
plt.ylim(bottom=-0.02) 
plt.xlim(left=1.5) 
plt.legend(loc='upper left')
plt.show()